In [195]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [196]:
import sqlite3
import pandas as pd

In [197]:
# Load data from CSV files
providers = pd.read_csv("/content/providers_data.csv")
receivers = pd.read_csv("/content/receivers_data.csv")
food_listings = pd.read_csv("/content/food_listings_data.csv")
claims = pd.read_csv("/content/claims_data.csv")

In [198]:
# Connect to SQLite (This will create a new database file if it doesn't exist)
connection = sqlite3.connect('food_management.db')

In [110]:
# Creates a database file 'food_management.db'
print("Connected to SQLite database!")

Connected to SQLite database!


In [199]:
 #Create a cursor object
cursor = connection.cursor()

In [200]:
# Create a PROVIDERS table
cursor.execute('''
CREATE TABLE IF NOT EXISTS providers (
    Provider_ID INTEGER PRIMARY KEY,
    Provider_Name TEXT,
    Type TEXT,
    Address TEXT,
    City TEXT,
    Contact TEXT)
''')

In [201]:
# Commit the changes (required to save)
connection.commit()
print("Table created successfully!")

Table created successfully!


In [202]:
#Creating RECEIVERS tables
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Receivers (
        Receiver_ID INTEGER PRIMARY KEY,
        Name TEXT,
        Type TEXT,
        City TEXT,
        Contact TEXT
    )
''')

In [203]:
# Commit the changes (required to save)
connection.commit()
print("Table created successfully!")

Table created successfully!


In [204]:
#Creating Food_listings tables
cursor.execute('''
    CREATE TABLE IF NOT EXISTS FOOD_LISTINGS (
        FOOD_ID INTEGER PRIMARY KEY,
        FOOD_Name TEXT,
        QUANITTY INTEGER,
        Expiry_Date DATE,
        Provider_ID INTEGER,
        Provider_Type TEXT,
        Location TEXT,
        food_Type TEXT,
        Meal_Type TEXT

    )
''')

In [205]:
# Commit the changes (required to save)
connection.commit()
print("Table created successfully!")

Table created successfully!


In [206]:
#Creating CLAIM tables
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Claims (
        CLAIM_ID INTEGER PRIMARY KEY,
        FOOD_ID INTEGER,
        Receiver_ID INTEGER,
        Status TEXT,
        Timestamp DATETIME

    )
''')

In [207]:
# Commit the changes (required to save)
connection.commit()
print("Table created successfully!")

Table created successfully!


In [208]:
# Iterate through the DataFrame and insert each row separately
# Replace or ignore if Provider_ID exists
for row in providers.values.tolist():
    try:
        cursor.execute('''
            INSERT INTO providers (Provider_ID, Provider_Name, Type, Address, City, Contact)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', row)
    except sqlite3.IntegrityError:
        # Handle the error: either ignore or update
        # For ignoring, you can use 'pass'
        # For updating, use an UPDATE query
        pass  # Ignore the duplicate Provider_ID error

connection.commit()

In [209]:
# Commit the changes (required to save)
connection.commit()
print("Table values insert successfully!")

Table values insert successfully!


In [212]:
for row in receivers.values.tolist():
    try:
      cursor.execute('''
            INSERT INTO receivers (Receiver_ID,Name, Type, City, Contact)
            VALUES (?, ?, ?, ?, ?)
        ''', row)
    except sqlite3.IntegrityError:
        # Handle the error: either ignore or update
        # For ignoring, you can use 'pass'
        # For updating, use an UPDATE query
        pass  # Ignore the duplicate Provider_ID error
connection.commit()
print("Table values insert successfully!")


Table values insert successfully!


In [213]:
for row in food_listings.values.tolist():
    try:
      cursor.execute('''
            INSERT INTO Food_listings (FOOD_ID,FOOD_Name,QUANITTY,Expiry_Date,Provider_ID , Provider_Type , Location ,food_Type ,Meal_Type)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', row)
    except sqlite3.IntegrityError:
        # Handle the error: either ignore or update
        # For ignoring, you can use 'pass'
        # For updating, use an UPDATE query
        pass  # Ignore the duplicate Provider_ID error
# Commit the changes (required to save)
connection.commit()
print("Table values insert successfully!")

Table values insert successfully!


In [214]:
for row in claims.values.tolist():
    try:
      cursor.execute('''
            INSERT INTO Claims (CLAIM_ID,FOOD_ID,receiver_ID,status,Timestamp)
           VALUES ( ?, ?, ?, ?, ?)
        ''', row)
    except sqlite3.IntegrityError:
        # Handle the error: either ignore or update
        # For ignoring, you can use 'pass'
        # For updating, use an UPDATE query
        pass  # Ignore the duplicate Provider_ID error
# Commit the changes (required to save)
connection.commit()
print("Table values insert successfully!")

Table values insert successfully!


In [229]:
# Define the update query
cursor.execute("""
UPDATE providers
SET Contact = 'New Contact Info'  -- Replace 'New Contact Info' with the actual value
WHERE Provider_ID = 1;  -- Replace 1 with the actual Provider_ID
""")

# Commit the changes to the database
connection.commit()
print("Data updated successfully!")

Data updated successfully!


In [230]:
#delete operation query
cursor.execute( """
DELETE FROM providers
WHERE Provider_ID = 0;  -- Replace 1 with the actual Provider_ID
""")

# Commit the changes to the database
connection.commit()

print("Data deleted successfully!")

Data deleted successfully!


In [231]:
# insert operation query
cursor.execute("SELECT 1 FROM providers WHERE Provider_ID = ?", (123,))
existing_provider = cursor.fetchone()

if not existing_provider:
    cursor.execute("""
INSERT INTO providers (Provider_ID, Provider_Name, Type, Address, City, Contact)
VALUES (123, 'New Provider', 'Restaurant', '123 Main St', 'Anytown', '555-1234');
""")
    # Commit the changes to the database
    connection.commit()
    print("Data added successfully!")
else:
    print("Provider with ID 123 already exists. Skipping insertion.")

Provider with ID 123 already exists. Skipping insertion.


In [253]:
#myself 1.which city have most highest food provider?
cursor.execute("""
SELECT City, Provider_Name, SUM(QUANITTY) AS TotalQuantity
FROM FOOD_LISTINGS
JOIN providers ON FOOD_LISTINGS.Provider_ID = providers.Provider_ID
GROUP BY City, Provider_Name
ORDER BY TotalQuantity DESC
LIMIT 1;
""")
result = cursor.fetchone()

# Print the result
if result:
    city, provider_name, total_quantity = result
    print(f"The city with the highest food provider is {city}, with {provider_name} donating a total quantity of {total_quantity}.")
else:
    print("No food listings found in the database.")

The city with the highest food provider is South Kathryn, with Barry Group donating a total quantity of 179.


In [215]:
#myself 2.list out the food providers?
#show the insert the values of provider table
cursor.execute("SELECT * FROM providers")
provider = cursor.fetchall()

# Get column names from cursor description
headers = [i[0] for i in cursor.description]
for row in provider:
    from tabulate import tabulate
print(tabulate(provider,headers=[i[0] for i in cursor.description],tablefmt='psql'))

+---------------+-----------------------------------+------------------+------------------------------------+--------------------------+------------------------+
|   Provider_ID | Provider_Name                     | Type             | Address                            | City                     | Contact                |
|---------------+-----------------------------------+------------------+------------------------------------+--------------------------+------------------------|
|             1 | Gonzales-Cochran                  | Supermarket      | 74347 Christopher Extensions       | New Jessica              | New Contact Info       |
|               |                                   |                  | Andreamouth, OK 91839              |                          |                        |
|             2 | Nielsen, Johnson and Fuller       | Grocery Store    | 91228 Hanson Stream                | East Sheena              | +1-925-283-8901x6297   |
|               |           

In [216]:
#myself 3.list out the food receivers ?
#show the insert the values of receiver table
cursor.execute("SELECT * FROM receivers")
receiver = cursor.fetchall()

# Get column names from cursor description
headers = [i[0] for i in cursor.description]
for row in provider:
    from tabulate import tabulate
print(tabulate(receiver,headers=[i[0] for i in cursor.description],tablefmt='psql'))

+---------------+-------------------------+------------+------------------------+------------------------+
|   Receiver_ID | Name                    | Type       | City                   | Contact                |
|---------------+-------------------------+------------+------------------------+------------------------|
|             1 | Donald Gomez            | Shelter    | Port Carlburgh         | (955)922-5295          |
|             2 | Laurie Ramos            | Individual | Lewisburgh             | 761.042.1570           |
|             3 | Ashley Mckee            | NGO        | South Randalltown      | 691-023-0094x856       |
|             4 | Erika Rose              | NGO        | South Shaneville       | 8296491111             |
|             5 | John Romero             | Individual | Bakerport              | 067.491.0154           |
|             6 | Mandy Sutton PhD        | Shelter    | East Sharimouth        | 682-777-5357           |
|             7 | Kenneth Baker      

In [217]:
#myself 4.list out the food_listings?
#show the insert the values of food_listings table
cursor.execute("SELECT * FROM food_listings")
food_listings = cursor.fetchall()

# Get column names from cursor description
headers = [i[0] for i in cursor.description]
for row in food_listings:
    from tabulate import tabulate
print(tabulate(food_listings,headers=[i[0] for i in cursor.description],tablefmt='psql'))

+-----------+-------------+------------+---------------+---------------+------------------+--------------------------+----------------+-------------+
|   FOOD_ID | FOOD_Name   |   QUANITTY | Expiry_Date   |   Provider_ID | Provider_Type    | Location                 | food_Type      | Meal_Type   |
|-----------+-------------+------------+---------------+---------------+------------------+--------------------------+----------------+-------------|
|         1 | Bread       |         43 | 3/17/2025     |           110 | Grocery Store    | South Kellyville         | Non-Vegetarian | Breakfast   |
|         2 | Soup        |         22 | 3/24/2025     |           791 | Grocery Store    | West James               | Non-Vegetarian | Dinner      |
|         3 | Fruits      |         46 | 3/28/2025     |           478 | Catering Service | Lake Regina              | Vegan          | Breakfast   |
|         4 | Fruits      |         15 | 3/16/2025     |           930 | Restaurant       | Kellytow

In [218]:
#myself 5.list out the claims?
#show the insert the values of claims table
cursor.execute("SELECT * FROM claims")
claims = cursor.fetchall()

# Get column names from cursor description
headers = [i[0] for i in cursor.description]
for row in claims:
    from tabulate import tabulate
print(tabulate(claims,headers=[i[0] for i in cursor.description],tablefmt='psql'))

+------------+-----------+---------------+-----------+-----------------+
|   CLAIM_ID |   FOOD_ID |   Receiver_ID | Status    | Timestamp       |
|------------+-----------+---------------+-----------+-----------------|
|          1 |       164 |           908 | Pending   | 3/5/2025 5:26   |
|          2 |       353 |           391 | Cancelled | 3/11/2025 10:24 |
|          3 |       626 |           492 | Completed | 3/21/2025 0:59  |
|          4 |        61 |           933 | Cancelled | 3/4/2025 9:08   |
|          5 |       345 |           229 | Pending   | 3/14/2025 15:17 |
|          6 |       273 |           607 | Cancelled | 3/2/2025 2:50   |
|          7 |       735 |           256 | Cancelled | 3/7/2025 23:58  |
|          8 |       382 |           900 | Pending   | 3/5/2025 7:07   |
|          9 |       278 |           807 | Completed | 3/18/2025 2:26  |
|         10 |       669 |           975 | Completed | 3/6/2025 19:57  |
|         11 |       290 |           118 | Pending 

In [228]:
#2.Which type of food provider (restaurant, grocery store, etc.) contributes the most food?
cursor.execute("""
SELECT Provider_Type, SUM(QUANITTY) AS TotalQuantity
FROM FOOD_LISTINGS
GROUP BY Provider_Type
ORDER BY TotalQuantity DESC
LIMIT 1;
""")
result = cursor.fetchone()

# Print the result
if result:
    provider_type, total_quantity = result
    print(f"The food provider type that contributes the most food is '{provider_type}' with a total quantity of {total_quantity}.")
else:
    print("No food listings found in the database.")

connection.commit()

The food provider type that contributes the most food is 'Restaurant' with a total quantity of 6923.


In [263]:
#1.How many food providers and receivers are there in each city?
cursor.execute("""
SELECT
    T1.City,  -- Specify the table for City column (providers table in this case)
    COUNT(DISTINCT CASE WHEN T1.Type = 'Restaurant' THEN T1.Provider_ID ELSE NULL END) as num_providers,
    COUNT(DISTINCT T2.Receiver_ID) as num_receivers
FROM providers AS T1
LEFT JOIN Receivers AS T2 ON T1.City = T2.City
GROUP BY T1.City; -- Also specify the table for City in the GROUP BY clause
""")
results = cursor.fetchall()

print("Number of Food Providers and Receivers in Each City:")
print("-" * 40)  # Print a separator line
print("{:<20} {:<15} {:<15}".format("City", "Providers", "Receivers"))
print("-" * 40)

for city, num_providers, num_receivers in results:
 print("{:<20} {:<15} {:<15}".format(city, num_providers, num_receivers))

Number of Food Providers and Receivers in Each City:
----------------------------------------
City                 Providers       Receivers      
----------------------------------------
Adambury             0               0              
Adamsview            1               0              
Adamsville           1               0              
Aguirreville         0               0              
Alexanderchester     0               0              
Alexanderstad        1               0              
Allenborough         0               1              
Allenton             0               0              
Amandaborough        1               0              
Amandashire          0               0              
Amberton             1               0              
Ambertown            0               0              
Amyport              0               0              
Andersonmouth        0               0              
Andersonville        0               0              
Andreaborough    

In [223]:
#3.What is the contact information of food providers in a specific city?
cursor.execute("""
SELECT Provider_Name, Contact
FROM providers
WHERE City = ?;
""", ('Williamsfort',)) # Assuming you want to filter by 'Williamsfort'
# Fetch the results
result = cursor.fetchall()
for row in result:
    provider_name, contact = row
    print(f"Providers: {provider_name}, Contact: {contact}")

Providers: Johnson Group, Contact: (007)484-4365x34610


In [224]:
#4.Which receivers have claimed the most food?
cursor.execute( """
SELECT
    r.Name,
    COUNT(c.CLAIM_ID) AS TotalClaims
FROM
    Receivers r
JOIN
    Claims c ON r.Receiver_ID = c.Receiver_ID
GROUP BY
    r.Name
ORDER BY
    TotalClaims DESC
LIMIT 1; -- To get only the receiver with the most claims
""")
# Fetch the results
result = cursor.fetchone()  # Get the first row (receiver with most claims)

# Print the result
if result:
    receiver_name, total_claims = result
    print(f"Receiver with the most claims: {receiver_name} ({total_claims} claims)")
else:
    print("No claims found in the database.")

Receiver with the most claims: William Frederick (5 claims)


In [178]:
#5.What is the total quantity of food available from all providers?
cursor.execute( """
SELECT SUM(QUANITTY) AS TotalFoodQuantity
FROM FOOD_LISTINGS;
""")
result = cursor.fetchone()

# Print the result
if result:
    total_quantity = result[0]  # Get the first element (TotalFoodQuantity) from the result tuple
    print(f"Total quantity of food available from all providers: {total_quantity}")
else:
    print("No food listings found in the database.")

Total quantity of food available from all providers: 25794


In [179]:
#6.Which city has the highest number of food listings?
cursor.execute( """
SELECT Location, COUNT(*) AS ListingCount
FROM FOOD_LISTINGS
GROUP BY Location
ORDER BY ListingCount DESC
LIMIT 1;
""")
result = cursor.fetchone()

# Print the result
if result:
    city, listing_count = result
    print(f"The city with the highest number of food listings is {city} with {listing_count} listings.")
else:
    print("No food listings found in the database.")


The city with the highest number of food listings is South Kathryn with 6 listings.


In [180]:
#7.What are the most commonly available food types?
cursor.execute("""
SELECT food_Type, COUNT(*) AS FoodTypeCount
FROM FOOD_LISTINGS
GROUP BY food_Type
ORDER BY FoodTypeCount DESC
LIMIT 5;  -- Change the limit to get more or fewer top food types
""")
results = cursor.fetchall()
# Print the results
print("Most commonly available food types:")
for row in results:
    food_type, food_type_count = row
    print(f" {food_type}: {food_type_count} listings")


Most commonly available food types:
 Vegetarian: 336 listings
 Vegan: 334 listings
 Non-Vegetarian: 330 listings


In [243]:
#8.How many food claims have been made for each food item?
cursor.execute("""
SELECT
    fl.FOOD_Name,
    COUNT(c.CLAIM_ID) AS ClaimCount
FROM
    FOOD_LISTINGS fl
LEFT JOIN
    Claims c ON fl.FOOD_ID = c.FOOD_ID
GROUP BY
    fl.FOOD_Name
ORDER BY
    ClaimCount DESC;
""")

results = cursor.fetchall()
print("Food Item Claims:")
for row in results:
    food_name, claim_count = row
    print(f"- {food_name}: {claim_count} claims")



Food Item Claims:
- Rice: 122 claims
- Soup: 114 claims
- Dairy: 110 claims
- Fish: 108 claims
- Salad: 106 claims
- Chicken: 102 claims
- Bread: 94 claims
- Pasta: 87 claims
- Vegetables: 86 claims
- Fruits: 71 claims


In [244]:
#9.Which provider has had the highest number of successful food claims?
cursor.execute("""
SELECT
    p.Provider_Name,
    COUNT(c.CLAIM_ID) AS TotalSuccessfulClaims
FROM
    providers p
JOIN
    FOOD_LISTINGS fl ON p.Provider_ID = fl.Provider_ID
JOIN
    Claims c ON fl.FOOD_ID = c.FOOD_ID
WHERE
    c.Status = 'claimed'  -- Filter for successful claims (assuming 'claimed' indicates success)
GROUP BY
    p.Provider_Name
ORDER BY
    TotalSuccessfulClaims DESC
LIMIT 1; -- To get only the provider with the highest claims
""")

result = cursor.fetchone()

# Print the result
if result:
    provider_name, total_successful_claims = result
    print(f"Provider with the highest number of successful claims: {provider_name} ({total_successful_claims} claims)")
else:
    print("No successful claims found in the database.")

No successful claims found in the database.


In [247]:
#10.What percentage of food claims are completed vs. pending vs. canceled?
cursor.execute("""
SELECT Status, COUNT(*) AS StatusCount
FROM Claims
GROUP BY Status;
""")
results = cursor.fetchall()

# Calculate the total number of claims
total_claims = sum(row[1] for row in results)

# Print the percentages for each status
print("Claim Status Percentages:")
for row in results:
    status, status_count = row
    percentage = (status_count / total_claims) * 100
    print(f"- {status}: {percentage:.4f}%")

Claim Status Percentages:
- Cancelled: 33.6000%
- Completed: 33.9000%
- Pending: 32.5000%


In [248]:
#11.What is the average quantity of food claimed per receiver?
cursor.execute("""
SELECT CAST(SUM(fl.QUANITTY) AS REAL) / COUNT(DISTINCT c.Receiver_ID) AS AverageQuantity
FROM Claims c
JOIN FOOD_LISTINGS fl ON c.FOOD_ID = fl.FOOD_ID;
""")
result = cursor.fetchone()

# Print the average quantity
if result:
    average_quantity = result[0]
    print(f"Average quantity of food claimed per receiver: {average_quantity:.4f}")
else:
    print("No claims found in the database.")

Average quantity of food claimed per receiver: 41.6010


In [249]:
#12.Which meal type (breakfast, lunch, dinner, snacks) is claimed the most?
cursor.execute("""
SELECT Meal_Type, COUNT(*) AS ClaimCount
FROM FOOD_LISTINGS
JOIN Claims ON FOOD_LISTINGS.FOOD_ID = Claims.FOOD_ID
GROUP BY Meal_Type
ORDER BY ClaimCount DESC
LIMIT 1;
""")
result = cursor.fetchone()

# Print the result
if result:
    meal_type, claim_count = result
    print(f"The most claimed meal type is: {meal_type} ({claim_count} claims)")
else:
    print("No claims found for any meal type.")

The most claimed meal type is: Breakfast (278 claims)


In [251]:
#13.What is the total quantity of food donated by each provider?
cursor.execute("""
SELECT p.Provider_Name, SUM(fl.QUANITTY) AS TotalDonatedQuantity
FROM providers p
JOIN FOOD_LISTINGS fl ON p.Provider_ID = fl.Provider_ID
GROUP BY p.Provider_Name
ORDER BY TotalDonatedQuantity DESC;
""")
results = cursor.fetchall()
print("Total Quantity of Food Donated by Each Provider:")# Fetch and print the results
for provider_name, total_quantity in results:
    print(f"- {provider_name}: {total_quantity}")

Total Quantity of Food Donated by Each Provider:
- Miller Inc: 217
- Barry Group: 179
- Evans, Wright and Mitchell: 158
- Smith Group: 150
- Campbell LLC: 145
- Nelson LLC: 142
- Ruiz-Oneal: 140
- Blankenship-Lewis: 124
- Kelly-Ware: 123
- Bradford-Martinez: 121
- Shepherd and Sons: 116
- Hampton-Lee: 116
- Jones, Ortega and Rubio: 115
- Ortiz-Lee: 114
- Johnson-Ray: 113
- Barker LLC: 110
- Moore Group: 106
- Hill, Davis and Stewart: 106
- Steele Ltd: 104
- Butler-Richardson: 104
- Lopez, Roach and Roach: 102
- Hunter, Ballard and Caldwell: 101
- Carter-Jones: 101
- Hogan-Johnston: 99
- Cox LLC: 99
- Baker-Mcdonald: 99
- Wong-Reese: 98
- Jackson Ltd: 98
- Aguilar-Frederick: 98
- Schmidt-Alexander: 97
- Phillips, Wolfe and Martin: 97
- Garcia-Hunter: 97
- Rogers, Harmon and Gordon: 96
- Jones, Rojas and Brown: 96
- Flores-Wade: 96
- Clark, Prince and Williams: 96
- Brown-Stephens: 96
- Carey PLC: 95
- Wilson Group: 94
- Lambert Ltd: 94
- Miller Ltd: 93
- Hudson, Spence and Perez: 93
- D

In [264]:
pip install streamlit


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.1 MB/s eta 0:00:00
